# Lektion 13: LangGraph – State, Nodes & Routing



| | |
|---|---|
| **Zeit** | ~75 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 01–12 |
| **Du baust** | Deine ersten drei LangGraph-Workflows: gerades Band, Verzweigung (Weiche) und Schleife mit Bremse |

## 🎯 Nach dieser Lektion kannst du:

- erklären, warum Chains (gerade Linien) für Verzweigungen nicht reichen

- einen **State** (gemeinsamer Notizblock) definieren und Nodes Daten hineinschreiben lassen

- konditionale Kanten (die Weiche) einsetzen

- einen Loop mit **Doppel-Bedingung** bauen (Qualität ODER Limit)

- das Diagramm → Code → Ausführung-Dreieck selbst erleben

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Das Bahnhofnetz

**WARUM:** Chains (Lektion 07+) sind **gerade Linien**: Input → Verarbeitung → Output, immer vorwärts. Aber dein Build-3-Assistent muss **verzweigen** (Vektor- oder Graph-Suche?) und **wiederholen** (Qualitätsprüfung mit Nachbesserung). Dafür brauchst du ein Werkzeug, das Entscheidungen und Schleifen *erste Klasse* behandelt.

**WAS:** **LangGraph** organisiert Arbeit als Graph. Die Vokabeln:

- **State** = der gemeinsame Notizbuch im Zug – jede Station **liest und schreibt** hinein
- **Node** = eine Station: genau eine Person erledigt genau eine Aufgabe
- **Edge** = ein Gleis von Station zu Station
- **Conditional Edge** = die Weiche: der Fahrdienstleiter liest den Notizblock und stellt die Weiche
- **Loop** = ein Rückgleis ("noch nicht gut genug → zurück!")

```text
START → klassifizieren → (Weiche)
                         ├─→ antwort_rechnung ──┐
                         ├─→ antwort_technisch ─┼→ END
                         └─→ antwort_allgemein ──┘
```

## Schritt 1: Der erste Graph (Hello Graph)

**WAS:** Ein State ist ein `TypedDict` (Lektion 01: Dictionary mit fester Form!). **WICHTIG:** Nodes geben nur ihre **Änderungen** zurück – LangGraph mischt sie in den State (Merge).

In [2]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict

class Zustand(TypedDict):
    begruessung: str

def begruessen(state: Zustand) -> dict:
    return {"begruessung": "Hallo LangGraph!"}

graph = StateGraph(Zustand)
graph.add_node("begruessen", begruessen)
graph.add_edge(START, "begruessen")
graph.add_edge("begruessen", END)

app = graph.compile()          # kompilieren -> ausfuehrbar
ergebnis = app.invoke({"begruessung": ""})
print(ergebnis)

{'begruessung': 'Hallo LangGraph!'}


### 🔎 Was sehen wir hier?
1. Schema (`TypedDict`) → Node-Funktion → Kanten → `compile()` → `invoke()`.
2. Der Node gibt `{{"begruessung": ...}}` zurück – **nur die Änderung**, nicht den ganzen State.
3. Das Ergebnis von `invoke` ist der finale State. **TRY:** Füge einen zweiten Node hinzu, der den Text erweitert!

## Schritt 2: Die Weiche (konditionale Kanten)

Jetzt das Entscheidungs-Muster – ein kleiner Support-Router. **Die wichtige Trennung:** `klassifizieren` **schreibt** (Regel-Logik), die Weichen-Funktion **liest nur** (sie entscheidet, sie arbeitet nicht).

In [3]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict

class TicketZustand(TypedDict):
    eingabe: str
    kategorie: str
    antwort: str

def klassifizieren(state: TicketZustand) -> dict:
    text = state["eingabe"].lower()
    if "rechnung" in text or "betrag" in text:
        return {"kategorie": "rechnung"}
    if any(w in text for w in ("funktioniert", "fehler", "absturz", "stuerzt")):
        return {"kategorie": "technisch"}
    return {"kategorie": "allgemein"}

def weiche(state: TicketZustand) -> str:
    """Der Fahrdienstleiter: liest NUR und nennt den naechsten Node."""
    return state["kategorie"]

def antwort_rechnung(state):
    return {"antwort": "Ihre Anfrage geht an die Buchhaltung (5 Werktage)."}
def antwort_technisch(state):
    return {"antwort": "Das Technik-Team nimmt sich dem an. Geraet + Fehlermeldung?"}
def antwort_allgemein(state):
    return {"antwort": "Danke! Das allgemeine Team antwortet in Kuerze."}

router = StateGraph(TicketZustand)
router.add_node("klassifizieren", klassifizieren)
for name, funktion in (("antwort_rechnung", antwort_rechnung),
                       ("antwort_technisch", antwort_technisch),
                       ("antwort_allgemein", antwort_allgemein)):
    router.add_node(name, funktion)
    router.add_edge(name, END)

router.add_edge(START, "klassifizieren")
router.add_conditional_edges(
    "klassifizieren", weiche,
    {"rechnung": "antwort_rechnung",
     "technisch": "antwort_technisch",
     "allgemein": "antwort_allgemein"},
)
router_app = router.compile()

for eingabe in ("Meine Rechnung hat einen falschen Betrag.",
                "Die App stuerzt seit heute ab.",
                "Wo sehe ich meine Arbeitszeiten?"):
    ergebnis = router_app.invoke({"eingabe": eingabe, "kategorie": "", "antwort": ""})
    print(f"{ergebnis['kategorie']:10s} -> {ergebnis['antwort']}")

rechnung   -> Ihre Anfrage geht an die Buchhaltung (5 Werktage).
technisch  -> Das Technik-Team nimmt sich dem an. Geraet + Fehlermeldung?
allgemein  -> Danke! Das allgemeine Team antwortet in Kuerze.


## ✏️ Deine Aufgabe: Dein eigener Router

Du baust eine Verzweigung mit eigener Regel-Logik – inklusive beweisbarem Test.

**Anforderungen:**

- Schreibe `klassifiziere_frage(frage)`, das `"vektor"`, `"graph"` oder `"hybrid"` zurückgibt: Verbindungsfragen („haengt", „wer", „zusammen") → `graph`; Gesamtfragen („gesamt", „alles", „vergleich") → `hybrid`; sonst `vektor`.

- Baue einen Mini-Graph (1 Klassifizier-Node + 3 Antwort-Nodes), der die Kategorie in `antwort` schreibt (z. B. `"Route: graph"`).

- Teste: „Was steht über Urlaub?“ → vektor; „Wer hängt am Projekt?“ → graph; „Was wissen wir gesamt?“ → hybrid.

**Schreibe deinen Code in die folgende Zelle:**

In [4]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict

class FrageZustand(TypedDict):
    frage: str
    route: str
    antwort: str

def klassifiziere_frage(frage):
    text = frage.lower()
    if any(w in text for w in ("haengt", "zusammen?", "wer ", "beziehung", "verbind")):
        return "graph"
    if any(w in text for w in ("gesamt", "alles", "vergleich", "sowohl")):
        return "hybrid"
    return "vektor"

g = StateGraph(FrageZustand)
g.add_node("klassifizieren", lambda s: {"route": klassifiziere_frage(s["frage"])})
for route in ("vektor", "graph", "hybrid"):
    g.add_node(f"antwort_{route}",
               lambda s, r=route: {"antwort": f"Route: {r}"})
    g.add_edge(f"antwort_{route}", END)
g.add_edge(START, "klassifizieren")
g.add_conditional_edges("klassifizieren", lambda s: s["route"],
    {r: f"antwort_{r}" for r in ("vektor", "graph", "hybrid")})
app = g.compile()

print(app.invoke({"frage": "Wer haengt am Projekt?", "route": "", "antwort": ""}))

{'frage': 'Wer haengt am Projekt?', 'route': 'graph', 'antwort': 'Route: graph'}


**💡 Warum diese Loesung funktioniert:**

Dies ist exakt die Klassifikation aus Build 3! Mit Key teilst du dem Lambda die Route mit (`r=route` im Default-Argument) – ein sauberer Trick, um drei Antwort-Nodes aus einer Fabrik zu bauen. Die Route-Entscheidung als **eigener Node** macht sie testbar – genau das prüft die Check-Zelle.

### ✅ Richtig, wenn:

- [ ] Die drei Testfragen landen auf den erwarteten Routen.

- [ ] Die Weichen-Funktion liest nur (keine State-Änderung in ihr!).

- [ ] Das Ergebnis des Graph enthält `route` und `antwort`.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [5]:
assert klassifiziere_frage("Was steht ueber Urlaub?") == "vektor"
assert klassifiziere_frage("Wer haengt am Projekt dran?") == "graph"
assert klassifiziere_frage("Was wissen wir gesamt ueber das Projekt?") == "hybrid"
ergebnis = app.invoke({"frage": "Wer haengt am Projekt?", "route": "", "antwort": ""})
assert ergebnis["route"] == "graph"
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Greife auf `frage.lower()` zu; prüfe die Schlüsselwörter mit `any(w in text for w in (...))`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Das Mapping-Dict in `add_conditional_edges` übersetzt Kategorie → Node-Name.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Der State braucht Felder: `frage`, `route`, `antwort`.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Warum ist „Klassifikation als eigener Node" besser, als die Entscheidung in die Antwort-Nodes zu verteilen?

2. Was wäre der Loop in einem echten LLM-Workflow – und was kostet er unbegrenzt?

## Schritt 3: Der Loop mit Bremse

**Metapher – die Küche mit Qualitätskontrolle:** Koch richtet den Teller an, Kontrolle prüft. Nicht schön? Zurück an den Herd! **Aber:** Nach drei Versuchen wird serviert – sonst wartet der Gast ewig. Der Versuchszähler im State ist die Bremse.

In [6]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict

class PolierZustand(TypedDict):
    text: str
    bewertung: int
    versuche: int
    max_versuche: int
    historie: list

def entwerfen(state):
    neu = state["versuche"] + 1
    text = f"{state['text']} (Politur {neu})"
    print(f"  entwerfen: Durchgang {neu}")
    return {"text": text, "versuche": neu,
            "historie": state["historie"] + [text]}   # NEUE Liste zurueckgeben!

def pruefen(state):
    bewertung = min(100, 30 + 25 * state["versuche"])
    print(f"  pruefen: Bewertung {bewertung}")
    return {"bewertung": bewertung}

def weiche_loop(state):
    if state["bewertung"] >= 60:          # Bedingung 1: Qualitaet
        return "fertig"
    if state["versuche"] >= state["max_versuche"]:   # Bedingung 2: LIMIT
        return "fertig"
    return "nachbessern"

g = StateGraph(PolierZustand)
g.add_node("entwerfen", entwerfen)
g.add_node("pruefen", pruefen)
g.add_edge(START, "entwerfen")
g.add_edge("entwerfen", "pruefen")
g.add_conditional_edges("pruefen", weiche_loop,
                        {"fertig": END, "nachbessern": "entwerfen"})
loop_app = g.compile()

ergebnis = loop_app.invoke({"text": "Erster Entwurf", "bewertung": 0,
                            "versuche": 0, "max_versuche": 5, "historie": []})
print("\nFinaler Text:", ergebnis["text"])
print("Historie:", ergebnis["historie"])

  entwerfen: Durchgang 1
  pruefen: Bewertung 55
  entwerfen: Durchgang 2
  pruefen: Bewertung 80

Finaler Text: Erster Entwurf (Politur 1) (Politur 2)
Historie: ['Erster Entwurf (Politur 1)', 'Erster Entwurf (Politur 1) (Politur 2)']


### 🔎 Was sehen wir hier?
**Zwei Endbedingungen** – Qualität ODER Limit. Ohne Limit kann ein Loop endlos laufen (Kosten!). Und: Listen im State gibst du als **neue Liste** zurück (`state["historie"] + [text]`) – nicht mutieren! **TRY:** Setze `max_versuche=1` – der Lauf terminiert kontrolliert nach einem Durchgang.

## ➡️ Weiter

**Weiter mit Lektion 14:** `notebooks/07_langgraph/14_hybrid_retrieval.ipynb` – Vektor- und Graph-Suche im direkten Duell (und die Kombination, die Build 3 ausmacht).